## Concept focus — Serialization safety and trust boundaries

Serialization is not only about format conversion; it is about trust. Some formats are data-only, while others can construct objects or execute dangerous behavior if used carelessly with untrusted input.

```text
untrusted input -> parser -> in-memory object

Safe-ish for data only: JSON, CSV
Risky when object loading is allowed: pickle, unsafe YAML loaders
```

### How to think about it
The first security question is always “who produced this data?” Treat deserialization as crossing a trust boundary. If the input is untrusted, prefer plain data formats and minimal loaders that do not execute arbitrary code paths.

### Visual references and further study
- [pickle documentation warning](https://docs.python.org/3/library/pickle.html)
- [json documentation](https://docs.python.org/3/library/json.html)
- [OWASP deserialization overview](https://owasp.org/www-community/vulnerabilities/Deserialization_of_untrusted_data)
- [PyYAML docs](https://pyyaml.org/wiki/PyYAMLDocumentation)

---

# Module 19 — The Standard Library, Files, and Serialization

## Exercise 19.4 — Five vulnerabilities. Exploit each, then fix it

Every function below is exploitable. For each: write the exploit input, run it,
observe the damage, then fix the function so the same input is harmless.
Do this on your own machine, in a temp directory. The point is to see the
attacks work once, because a vulnerability you have only read about is one you
will write again.
Run:  python ex04_security.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. `pathlib`

Use `Path` everywhere. `os.path` is string manipulation pretending to be file
handling.

```text
from pathlib import Path

p = Path("data") / "raw" / "input.csv"     # / is the join operator
p.name          # 'input.csv'
p.stem          # 'input'
p.suffix        # '.csv'
p.suffixes      # ['.csv']   -- ['.tar', '.gz'] for archive.tar.gz
p.parent        # Path('data/raw')
p.parents       # all ancestors, lazily
p.absolute()    # not resolved -- may contain '..'
p.resolve()     # canonical: symlinks followed, '..' removed
p.exists() / .is_file() / .is_dir() / .is_symlink()
p.stat().st_size / .st_mtime

p.read_text(encoding="utf-8")              # always name the encoding
p.write_text(data, encoding="utf-8")
p.read_bytes() / p.write_bytes(b"...")

p.mkdir(parents=True, exist_ok=True)       # both flags, almost always
p.unlink(missing_ok=True)
p.rename(other) / p.replace(other)         # replace OVERWRITES, rename may not
list(p.glob("*.csv")) / list(p.rglob("*.py"))
Path.home() / Path.cwd()
```


**`resolve()` versus `absolute()` matters for security.** A user-supplied path
like `../../etc/passwd` is only visible after resolution:

In [ ]:
target = (base / user_input).resolve()
if not target.is_relative_to(base.resolve()):    # 3.9+
    raise ValueError("path escapes the base directory")

That check is the difference between a file server and a directory traversal
vulnerability.

### Atomic writes, again

In [ ]:
tmp = path.with_suffix(path.suffix + ".tmp")     # SAME directory
tmp.write_text(data, encoding="utf-8")
tmp.replace(path)                                 # atomic within a filesystem

Module 07 covered this; it belongs in your fingers.

---

## Concept 2. Files and I/O

In [ ]:
with path.open("r", encoding="utf-8", newline="") as fh: ...
with path.open("rb") as fh: ...             # binary: no encoding, no newline

| Mode | Meaning |
|---|---|
| `r` `w` `a` | read, truncate-and-write, append |
| `x` | create, fail if it exists — the safe way to avoid clobbering |
| `+` | read *and* write |
| `b` | binary |

Three things worth knowing:

**`newline=""` for the `csv` module.** Without it, `\r\n` inside a quoted field
is translated and the file is corrupted. The `csv` docs say this and everyone
skips it.

**Iterating a file yields lines lazily.** `for line in fh:` reads a buffer at a
time, so it works on a file larger than memory. `fh.readlines()` does not.

**Text mode does newline translation and encoding.** Binary mode does neither.
If you are computing a hash, comparing bytes, or handling anything non-text, use
binary.

In [ ]:
import shutil, tempfile, os

shutil.copy2(src, dst)              # copies metadata too
shutil.move(src, dst)
shutil.rmtree(path)
shutil.disk_usage(path)

with tempfile.TemporaryDirectory() as td: ...       # cleaned up always
with tempfile.NamedTemporaryFile(delete=False) as f: ...

---

## Concept 3. `json`

In [ ]:
import json

json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False, default=str)
json.loads(text)
json.dump(obj, fh)      # to a file
json.load(fh)

**What JSON cannot represent, and what Python does about it:**

| Python | JSON | Round trip? |
|---|---|---|
| `dict` with str keys | object | yes |
| `dict` with int keys | object with **str** keys | **no** — `{1: "a"}` comes back `{"1": "a"}` |
| `tuple` | array | **no** — comes back as a `list` |
| `set`, `bytes`, `Decimal`, `datetime` | — | **no** — `TypeError` unless handled |
| `float('nan')`, `inf` | not valid JSON | Python emits `NaN` anyway, which other parsers reject |
| large `int` | number | yes, but many parsers lose precision above 2⁵³ |

In [ ]:
class Encoder(json.JSONEncoder):
    def default(self, obj: Any) -> Any:
        if isinstance(obj, datetime):
            return obj.isoformat()
        if isinstance(obj, Decimal):
            return str(obj)             # str, NOT float -- Module 03
        if isinstance(obj, set):
            return sorted(obj)
        return super().default(obj)

**`ensure_ascii=False`** keeps non-ASCII readable (`"café"` rather than
`"café"`) and produces smaller output. **`sort_keys=True`** makes output
deterministic, which matters for diffs, caching and content hashing.

---

## Concept 5. `sqlite3`

A full SQL database, in the standard library, with no server.

In [ ]:
import sqlite3

with sqlite3.connect("app.db") as conn:      # NOTE: commits, does NOT close
    conn.row_factory = sqlite3.Row           # dict-like rows
    conn.execute("PRAGMA foreign_keys = ON") # OFF by default!
    cur = conn.execute("SELECT * FROM users WHERE age > ?", (18,))
    for row in cur:
        print(row["name"])

**Always use parameters, never string formatting.**

In [ ]:
conn.execute(f"SELECT * FROM users WHERE name = '{name}'")   # SQL INJECTION
conn.execute("SELECT * FROM users WHERE name = ?", (name,))  # correct

This is not a style preference. `name = "'; DROP TABLE users; --"` is the entire
attack, and parameterisation makes it structurally impossible because the value
never becomes part of the statement.

Two sqlite-specific traps: `with conn:` is a **transaction** context manager,
not a closing one — it commits or rolls back and leaves the connection open.
And foreign key enforcement is **off** by default, so your constraints do
nothing until you turn the pragma on.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `pathlib`
- Section 2: Files and I/O
- Section 3: `json`
- Section 4: `csv`
- Section 5: `sqlite3`
- Section 6: `pickle`, and why not to use it
- Section 7: `datetime`, and the one rule
- Section 8: `re`, at working depth
- Section 9: `subprocess`, safely
- Section 10: The rest, in one table

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import hashlib
import os
import pickle
import sqlite3
import subprocess
import tempfile
import time
from pathlib import Path


# --- 1: SQL injection ---------------------------------------------------------

---

## `find_user_vulnerable`

_find user vulnerable_

In [ ]:
def find_user_vulnerable(conn: sqlite3.Connection, name: str) -> list[tuple]:
    query = f"SELECT id, name, email FROM users WHERE name = '{name}'"
    return conn.execute(query).fetchall()

---

## `count_matches_vulnerable`

_count matches vulnerable_

In [ ]:
def count_matches_vulnerable(pattern: str, path: str) -> str:
    return subprocess.run(
        f"grep -c '{pattern}' {path}", shell=True,
        capture_output=True, text=True,
    ).stdout

---

## `read_user_file_vulnerable`

_read user file vulnerable_

In [ ]:
def read_user_file_vulnerable(base: Path, filename: str) -> str:
    return (base / filename).read_text(encoding="utf-8")

---

## `load_session_vulnerable`

_load session vulnerable_

In [ ]:
def load_session_vulnerable(data: bytes) -> object:
    return pickle.loads(data)

---

## `check_token_vulnerable`

_check token vulnerable_

In [ ]:
def check_token_vulnerable(supplied: str, expected: str) -> bool:
    return supplied == expected

---

## `demo_timing`

A crude demonstration that == returns faster on an early mismatch.

In [ ]:
def demo_timing() -> None:
    """A crude demonstration that == returns faster on an early mismatch."""
    secret = "s" * 1000 + "X"
    trials = 2000
    for candidate, label in [("a" + "s" * 1000, "differs at position 0"),
                             ("s" * 1000 + "Y", "differs at position 1000")]:
        start = time.perf_counter()
        for _ in range(trials):
            candidate == secret            # noqa: B015
        elapsed = time.perf_counter() - start
        print(f"    {label:<28} {elapsed * 1e6 / trials:8.3f} us/comparison")

---

## `setup_db`

_setup db_

In [ ]:
def setup_db() -> sqlite3.Connection:
    conn = sqlite3.connect(":memory:")
    conn.executescript("""
        CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT, email TEXT);
        CREATE TABLE secrets (id INTEGER PRIMARY KEY, api_key TEXT);
        INSERT INTO users VALUES (1, 'ada', 'ada@example.com');
        INSERT INTO users VALUES (2, 'bo', 'bo@example.com');
        INSERT INTO secrets VALUES (1, 'sk-super-secret');
    """)
    return conn

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    print("1. SQL injection")
    conn = setup_db()
    print("    normal search:", find_user_vulnerable(conn, "ada"))
    print("    TODO: write the three exploit strings and run them here")

    print("\n5. timing")
    demo_timing()
    print("    the difference is small but measurable, and it is enough")

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.